# ML Pipeline — ETL-sourced Training Data

End-to-end training pipeline that loads fully prepared features directly from the **SQLite database** maintained by `etl.py`, instead of re-fetching and re-engineering raw data on every run.

Four models are trained and saved:

| Model | Objective | Purpose | Saved as |
|---|---|---|---|
| LightGBM | MAE (default) | Best-estimate forecast | `best_lgbm_model_bayesian_etl.pkl` |
| LightGBM conservative | Quantile α=0.95 | Minimise under-prediction risk | `best_lgbm_model_bayesian_conservative_etl.pkl` |
| XGBoost | squared-error | Best-estimate forecast | `best_xgb_model_bayesian_etl.pkl` |
| XGBoost conservative | Quantile α=0.95 | Minimise under-prediction risk | `best_xgb_model_bayesian_conservative_etl.pkl` |

All hyperparameters are tuned with `BayesSearchCV` using time-series-aware 5-fold cross-validation (`TimeSeriesSplit`).  
The `_etl` postfix distinguishes these models from those trained in the legacy CSV-based pipeline (`07_complete_ml_pipeline.ipynb`).

In [ ]:
import sys
import os

# Add the src directory to the system path to allow importing custom modules
sys.path.insert(0, os.path.abspath('../src'))

import warnings
warnings.filterwarnings('ignore')

# Enable autoreload to automatically reload modules when they are edited
%load_ext autoreload
%autoreload 2

import numpy as np
import pickle
import pandas as pd

from fetch_prepare_data import *
from train_model_predict import *

# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

## 1. Update & Load Data from Database

`update_database()` is idempotent:
- **First run** — seeds both tables from the Kaggle CSV and the Open-Meteo API.
- **Subsequent runs** — only fills the gap between the last stored timestamp and yesterday; usually completes in seconds.

After the update, `load_combined_data()` reads the pre-joined `energy_weather_combined` view — all engineered features are already stored in the DB, so no re-computation is needed here.

Two columns are dropped before training because they are informational, not predictive:
- `smard_forecast_mwh` — SMARD official forecast, kept in the DB for actual-vs-forecast comparison
- `data_source` — provenance label (`'kaggle'` / `'smard'`)

The target column `energy_demand_mwh` is renamed to `EnergyDemand` to match the existing pipeline conventions used throughout the other notebooks.

In [ ]:
from etl import update_database, get_connection, load_combined_data

# Ensure the database is up to date — no-op if already current
update_database()

# Load the pre-joined, feature-engineered dataset from the DB view
conn = get_connection()
df_raw = load_combined_data(conn)
conn.close()

# Drop non-predictive columns; rename target to match existing pipeline conventions
df_for_modeling = (
    df_raw
    .drop(columns=['smard_forecast_mwh', 'data_source'], errors='ignore')
    .rename(columns={'energy_demand_mwh': 'EnergyDemand'})
    .dropna()
    .reset_index(drop=True)
)

print(f"Dataset : {len(df_for_modeling):,} rows")
print(f"Range   : {df_for_modeling['time'].min().date()} – {df_for_modeling['time'].max().date()}")
print(f"Features: {df_for_modeling.shape[1] - 1}  |  Target: EnergyDemand")
display(df_for_modeling.head(3))

## 2. Train / Test Split

A **chronological split at 2025-01-01** is used for all four models:
- **Train** — 2019-01-08 to 2024-12-31
- **Test** — 2025-01-01 onwards (unseen future data)

This mirrors real-world deployment: the model is trained on historical patterns and evaluated on data it has never seen.

The split date is passed as a tz-aware `pd.Timestamp` because the `time` column retains `Europe/Berlin` timezone information from the database.

In [ ]:
SPLIT_DATE = pd.Timestamp('2025-01-01', tz='Europe/Berlin')

features_train, target_train, features_test, target_test = train_test_split_by_date(
    df_for_modeling,
    date_column='time',
    target_column='EnergyDemand',
    split_date=SPLIT_DATE,
)

print(f"Train : {len(features_train):,} rows  ({features_train.shape[1]} features)")
print(f"Test  : {len(features_test):,} rows")

## 3. LightGBM — Standard (Best-Estimate)

LightGBM is tuned with `BayesSearchCV` over a continuous hyperparameter space using `TimeSeriesSplit(n_splits=5)`. The default `objective='regression'` minimises MAE, producing an unbiased best-estimate forecast. Tree-based models do not require feature scaling, so no preprocessor pipeline is needed.

| Parameter | Search range |
|---|---|
| `n_estimators` | 50 – 500 |
| `learning_rate` | 0.01 – 0.30 |
| `max_depth` | 3 – 15 |

In [ ]:
from lightgbm import LGBMRegressor

param_lgbm = {
    'n_estimators':  (50, 500),
    'learning_rate': (0.01, 0.3),
    'max_depth':     (3, 15),
}

model_lgbm = LGBMRegressor(random_state=42, force_col_wise=True)
best_model_lgbm, best_params_lgbm = tune_model_bayesian(
    model_pipeline=model_lgbm,
    in_param_bayes=param_lgbm,
    in_features_train=features_train,
    in_target_train=target_train,
)
print(f"Best hyperparameters: {best_params_lgbm}")
print()

pred_lgbm = best_model_lgbm.predict(features_test)
print_scores('LightGBM', target_test, pred_lgbm)

save_model_to_pickle(best_model_lgbm, '../models/best_lgbm_model_bayesian_etl.pkl')
print("Saved → ../models/best_lgbm_model_bayesian_etl.pkl")

## 4. XGBoost — Standard (Best-Estimate)

XGBoost is tuned with the same Bayesian optimisation approach. The search space is extended with subsampling and column-sampling parameters that help control overfitting on the large training set.

| Parameter | Search range |
|---|---|
| `n_estimators` | 50 – 500 |
| `max_depth` | 3 – 15 |
| `learning_rate` | 0.01 – 0.30 |
| `subsample` | 0.5 – 1.0 |
| `colsample_bytree` | 0.5 – 1.0 |

In [ ]:
from xgboost import XGBRegressor

param_xgb = {
    'n_estimators':     (50, 500),
    'max_depth':        (3, 15),
    'learning_rate':    (0.01, 0.3),
    'subsample':        (0.5, 1.0),
    'colsample_bytree': (0.5, 1.0),
}

model_xgb = XGBRegressor(random_state=42)
best_model_xgb, best_params_xgb = tune_model_bayesian(
    model_pipeline=model_xgb,
    in_param_bayes=param_xgb,
    in_features_train=features_train,
    in_target_train=target_train,
)
print(f"Best hyperparameters: {best_params_xgb}")
print()

pred_xgb = best_model_xgb.predict(features_test)
print_scores('XGBoost', target_test, pred_xgb)

save_model_to_pickle(best_model_xgb, '../models/best_xgb_model_bayesian_etl.pkl')
print("Saved → ../models/best_xgb_model_bayesian_etl.pkl")

## 5. Conservative Models — Quantile Regression (α = 0.95)

Grid operators prefer **over-provisioning** to under-provisioning: the cost of having too much capacity is far lower than the cost of a shortage. Conservative models use **quantile regression** at α = 0.95, targeting the 95th percentile of the conditional demand distribution.

| Framework | Setting |
|---|---|
| LightGBM | `objective='quantile'`, `alpha=0.95` |
| XGBoost | `objective='reg:quantileerror'`, `quantile_alpha=0.95` |

At α = 0.95 the model is calibrated to exceed actual demand in ~95% of hours. The trade-off is a positive mean bias (systematic over-prediction). The **conservatism check** after each model quantifies this on the test set.

The same hyperparameter search spaces (`param_lgbm`, `param_xgb`) are reused — only the loss function changes.

In [ ]:
# ── LightGBM conservative ─────────────────────────────────────────────────
model_lgbm_conservative = LGBMRegressor(
    objective='quantile',
    alpha=0.95,   # the higher the alpha, the more conservative
    random_state=42,
    force_col_wise=True,
)
best_model_lgbm_conservative, best_params_lgbm_conservative = tune_model_bayesian(
    model_pipeline=model_lgbm_conservative,
    in_param_bayes=param_lgbm,
    in_features_train=features_train,
    in_target_train=target_train,
)
print(f"Best hyperparameters: {best_params_lgbm_conservative}")
print()

pred_lgbm_conservative = best_model_lgbm_conservative.predict(features_test)
print_scores('LightGBM conservative', target_test, pred_lgbm_conservative)

save_model_to_pickle(best_model_lgbm_conservative, '../models/best_lgbm_model_bayesian_conservative_etl.pkl')
print("Saved → ../models/best_lgbm_model_bayesian_conservative_etl.pkl")

# Conservatism check — fraction of hours where prediction >= actual (target: ~95%)
coverage_lgbm = (pred_lgbm_conservative >= target_test.values).mean()
overpred_lgbm = pred_lgbm_conservative - target_test.values
print(f"\nConservatism check  (target: ~95%)")
print(f"  Coverage (pred >= actual) : {coverage_lgbm:.1%}")
print(f"  Mean bias                 : {overpred_lgbm.mean():+,.0f} MWh")
print(f"  Under-predictions         : {(overpred_lgbm < 0).sum()} / {len(overpred_lgbm)} hours")

In [ ]:
# ── XGBoost conservative ──────────────────────────────────────────────────
model_xgb_conservative = XGBRegressor(
    objective='reg:quantileerror',
    quantile_alpha=0.95,   # the higher the alpha, the more conservative
    random_state=42,
)
best_model_xgb_conservative, best_params_xgb_conservative = tune_model_bayesian(
    model_pipeline=model_xgb_conservative,
    in_param_bayes=param_xgb,
    in_features_train=features_train,
    in_target_train=target_train,
)
print(f"Best hyperparameters: {best_params_xgb_conservative}")
print()

pred_xgb_conservative = best_model_xgb_conservative.predict(features_test)
print_scores('XGBoost conservative', target_test, pred_xgb_conservative)

save_model_to_pickle(best_model_xgb_conservative, '../models/best_xgb_model_bayesian_conservative_etl.pkl')
print("Saved → ../models/best_xgb_model_bayesian_conservative_etl.pkl")

# Conservatism check
coverage_xgb = (pred_xgb_conservative >= target_test.values).mean()
overpred_xgb = pred_xgb_conservative - target_test.values
print(f"\nConservatism check  (target: ~95%)")
print(f"  Coverage (pred >= actual) : {coverage_xgb:.1%}")
print(f"  Mean bias                 : {overpred_xgb.mean():+,.0f} MWh")
print(f"  Under-predictions         : {(overpred_xgb < 0).sum()} / {len(overpred_xgb)} hours")

## 6. Learning Curves

Learning curves compare training MAE vs. cross-validation MAE as the training set size grows. They help diagnose:
- **Overfitting** — large persistent gap between train and CV score
- **Underfitting** — both scores converge at a high MAE
- **Good fit** — the gap closes as training data increases

The conservative models are expected to show higher absolute MAE than their standard counterparts, because quantile loss at α=0.95 penalises under-predictions much more heavily than over-predictions, yielding a systematically biased but intentionally conservative estimator.

In [ ]:
plot_learning_curve(best_model_lgbm,              'LightGBM (standard)',     features_train, target_train)
plot_learning_curve(best_model_xgb,               'XGBoost (standard)',      features_train, target_train)
plot_learning_curve(best_model_lgbm_conservative, 'LightGBM (conservative)', features_train, target_train)
plot_learning_curve(best_model_xgb_conservative,  'XGBoost (conservative)',  features_train, target_train)